In [1]:
!pwd

/home/govlept1004/jupyter_home


In [2]:
# 폴더 구조 만들기
root_folder = "/home/govlept1004/jupyter_home/NUMBER"

In [3]:
!mkdir $root_folder/Number_devkit

mkdir: cannot create directory ‘/home/govlept1004/jupyter_home/NUMBER/Number_devkit’: File exists


In [4]:
!mkdir $root_folder/Number_devkit/Annotations
!mkdir $root_folder/Number_devkit/JPEGImages
!mkdir $root_folder/Number_devkit/ImageSets
!mkdir $root_folder/Number_devkit/ImageSets/Main

mkdir: cannot create directory ‘/home/govlept1004/jupyter_home/NUMBER/Number_devkit/Annotations’: File exists
mkdir: cannot create directory ‘/home/govlept1004/jupyter_home/NUMBER/Number_devkit/JPEGImages’: File exists
mkdir: cannot create directory ‘/home/govlept1004/jupyter_home/NUMBER/Number_devkit/ImageSets’: File exists
mkdir: cannot create directory ‘/home/govlept1004/jupyter_home/NUMBER/Number_devkit/ImageSets/Main’: File exists


In [5]:
# 데이터가 너무 많아서 4000장으로 줄이는 코드 (기본 csv파일에서 추출)

In [6]:
import random

# 파일 경로 설정
train_txt_path = "/home/govlept1004/jupyter_home/NUMBERdevkit/ImageSets/Main/train.txt"
val_txt_path = "/home/govlept1004/jupyter_home/NUMBERdevkit/ImageSets/Main/val.txt"

# train.txt와 val.txt 파일을 읽어서 이미지 경로를 리스트에 저장
def read_image_paths(txt_path):
    with open(txt_path, "r") as f:
        image_paths = [line.strip() for line in f.readlines()]
    return image_paths

# 이미지 경로를 불러오기
train_images = read_image_paths(train_txt_path)
val_images = read_image_paths(val_txt_path)

# 전체 이미지 수 확인
print(f"전체 train 이미지 수: {len(train_images)}")
print(f"전체 val 이미지 수: {len(val_images)}")

# 4,000개의 이미지를 무작위로 샘플링
train_sampled = random.sample(train_images, 4000)  # train에서 4000개 샘플링
val_sampled = random.sample(val_images, 400)  # val에서 400개 샘플링 (필요에 따라 수정)

# 샘플링된 이미지 리스트 확인
print(f"샘플링된 train 이미지 수: {len(train_sampled)}")
print(f"샘플링된 val 이미지 수: {len(val_sampled)}")

# 새로운 train.txt, val.txt 파일로 저장
def save_image_paths(image_paths, output_txt_path):
    with open(output_txt_path, "w") as f:
        for path in image_paths:
            f.write(path + "\n")

# 새로운 txt 파일 저장
save_image_paths(train_sampled, "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/ImageSets/Main/train.txt")
save_image_paths(val_sampled, "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/ImageSets/Main/val.txt")


전체 train 이미지 수: 26721
전체 val 이미지 수: 6681
샘플링된 train 이미지 수: 4000
샘플링된 val 이미지 수: 400


In [7]:
import shutil
import os

# 경로 설정
train_txt_path = "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/ImageSets/Main/train.txt"
val_txt_path = "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/ImageSets/Main/val.txt"
image_base_path = "/home/govlept1004/jupyter_home/NUMBERdevkit/JPEGImages"  # 이미지 파일이 있는 폴더
annotations_base_path = "/home/govlept1004/jupyter_home/NUMBERdevkit/Annotations"  # XML 파일이 있는 폴더
train_dest_dir = "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/JPEGImages"  # train 이미지 복사 폴더
val_dest_dir = "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/JPEGImages"  # val 이미지 복사 폴더
train_annotations_dir = "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/Annotations"  # train XML 복사 폴더
val_annotations_dir = "/home/govlept1004/jupyter_home/NUMBER/Number_devkit/Annotations"  # val XML 복사 폴더

# 새로 생성된 train.txt, val.txt에서 이미지 파일 이름을 읽어오기
def read_image_paths(txt_path):
    with open(txt_path, "r") as f:
        image_paths = [line.strip() for line in f.readlines()]
    return image_paths

# 이미지 경로 불러오기
train_images = read_image_paths(train_txt_path)
val_images = read_image_paths(val_txt_path)

# 이미지 및 XML 파일 복사 함수
def copy_files(image_paths, img_dest_dir, anno_dest_dir):
    for img_filename in image_paths:
        # 이미지 파일 경로
        img_path = os.path.join(image_base_path, img_filename + ".png")  # 이미지 파일 확장자 맞추기
        xml_path = os.path.join(annotations_base_path, img_filename + ".xml")  # XML 파일 경로
        
        # 이미지 복사
        if os.path.exists(img_path):
            shutil.copy(img_path, os.path.join(img_dest_dir, os.path.basename(img_path)))
        
        # XML 복사
        if os.path.exists(xml_path):
            shutil.copy(xml_path, os.path.join(anno_dest_dir, os.path.basename(xml_path)))

# train 이미지 및 XML 파일 복사
copy_files(train_images, train_dest_dir, train_annotations_dir)

# val 이미지 및 XML 파일 복사
copy_files(val_images, val_dest_dir, val_annotations_dir)

print("✅ 이미지와 XML 파일 복사가 완료되었습니다.")

✅ 이미지와 XML 파일 복사가 완료되었습니다.


In [8]:
# 폴더 구조가 만들어졌으니 이제 TFRecord로 변환시켜보아요! 제공된 코드를 사용하면되요!

import os
import tensorflow as tf
import xml.etree.ElementTree as ET

# 클래스 이름을 문자열로 1~10까지 정확히 지정
VOC_CLASSES = [
    "1","2","3","4","5","6","7","8","9","0"
]
class_name_to_id = {name: i for i, name in enumerate(VOC_CLASSES)}

# XML 파싱 함수
def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.find("filename").text
    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    bboxes = []
    labels = []

    for obj in root.findall("object"):
        name = obj.find("name").text
        label = class_name_to_id[name]  # 문자열 그대로 매핑

        bbox = obj.find("bndbox")
        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))

        bboxes.append([xmin, ymin, xmax, ymax])
        labels.append(label)

    return filename, width, height, bboxes, labels

# TFRecord 예제 생성
def create_tf_example(xml_path, image_dir):
    filename, width, height, bboxes, labels = parse_voc_xml(xml_path)
    image_path = os.path.join(image_dir, filename)  # XML에 명시된 파일명을 그대로 사용

    with tf.io.gfile.GFile(image_path, 'rb') as fid:
        encoded_image = fid.read()

    xmins = [box[0] / width for box in bboxes]
    ymins = [box[1] / height for box in bboxes]
    xmaxs = [box[2] / width for box in bboxes]
    ymaxs = [box[3] / height for box in bboxes]

    classes_text = [VOC_CLASSES[label].encode('utf8') for label in labels]
    classes = labels

    feature = {
        'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[encoded_image])),
        'image/filename': tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename.encode('utf8')])),
        'image/height': tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
        'image/width': tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
        'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=xmins)),
        'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=ymins)),
        'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=xmaxs)),
        'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=ymaxs)),
        'image/object/class/text': tf.train.Feature(bytes_list=tf.train.BytesList(value=classes_text)),
        'image/object/class/label': tf.train.Feature(int64_list=tf.train.Int64List(value=classes)),
    }

    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example

# TFRecord 생성 함수
def generate_tfrecord(voc_dir, split_txt, output_path):
    annotation_dir = os.path.join(voc_dir, 'Annotations')
    image_dir = os.path.join(voc_dir, 'JPEGImages')

    with tf.io.TFRecordWriter(output_path) as writer:
        with open(split_txt, 'r') as f:
            lines = f.read().strip().splitlines()

        for img_id in lines:
            xml_path = os.path.join(annotation_dir, f"{img_id}.xml")

            if os.path.exists(xml_path):
                try:
                    example = create_tf_example(xml_path, image_dir)
                    writer.write(example.SerializeToString())
                except Exception as e:
                    print(f"[Error] {img_id} 처리 중 오류 발생: {e}")

VOC_ROOT_DIR = '/home/govlept1004/jupyter_home/NUMBER/Number_devkit'

generate_tfrecord(
    voc_dir=VOC_ROOT_DIR,
    split_txt=os.path.join(VOC_ROOT_DIR, "ImageSets/Main/train.txt"),
    output_path=os.path.join(VOC_ROOT_DIR, "number_train.tfrecord")
)

generate_tfrecord(
    voc_dir=VOC_ROOT_DIR,
    split_txt=os.path.join(VOC_ROOT_DIR, "ImageSets/Main/val.txt"),
    output_path=os.path.join(VOC_ROOT_DIR, "number_val.tfrecord")
)

In [9]:
# TFRecord가 생성 후 tf.data.Dataset 생성

IMAGE_SIZE = 128

def parse_tfrecord(example):
    features = {
        'image/encoded': tf.io.FixedLenFeature([], tf.string),
        'image/filename': tf.io.FixedLenFeature([], tf.string),
        'image/height': tf.io.FixedLenFeature([], tf.int64),
        'image/width': tf.io.FixedLenFeature([], tf.int64),
        'image/object/bbox/xmin': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/ymin': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/xmax': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/ymax': tf.io.VarLenFeature(tf.float32),
        'image/object/class/label': tf.io.VarLenFeature(tf.int64),
    }

    parsed = tf.io.parse_single_example(example, features)
    image = tf.image.decode_jpeg(parsed['image/encoded'], channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)

    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    xmins = tf.sparse.to_dense(parsed['image/object/bbox/xmin'])
    ymins = tf.sparse.to_dense(parsed['image/object/bbox/ymin'])
    xmaxs = tf.sparse.to_dense(parsed['image/object/bbox/xmax'])
    ymaxs = tf.sparse.to_dense(parsed['image/object/bbox/ymax'])
    labels = tf.sparse.to_dense(parsed['image/object/class/label'])

    boxes = tf.stack([xmins, ymins, xmaxs, ymaxs], axis=-1)

    return {
        "images": image,
        "bounding_boxes": {
            "boxes": boxes,
            "classes": tf.cast(labels, tf.int32),
        }
    }

def load_dataset(tfrecord_path, batch_size=8):
    dataset = tf.data.TFRecordDataset(tfrecord_path)
    dataset = dataset.map(parse_tfrecord, 
                          num_parallel_calls=tf.data.AUTOTUNE)
    
    # padded_batch 필수
    dataset = dataset.shuffle(1000).padded_batch(
        batch_size,
        padded_shapes={
            "images": [IMAGE_SIZE, IMAGE_SIZE, 3],
            "bounding_boxes": {
                "boxes": [None, 4],
                "classes": [None]
            }
        },
        drop_remainder=True
    ).prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = load_dataset("/home/govlept1004/jupyter_home/NUMBER/Number_devkit/number_train.tfrecord", 
                        batch_size=8)

val_ds = load_dataset("/home/govlept1004/jupyter_home/NUMBER/Number_devkit/number_val.tfrecord", 
                        batch_size=8)

In [10]:
# 재학습에 필요한 Dataset 준비 완료!!

# Model 생성
from keras_cv.models import RetinaNet

model = RetinaNet.from_preset(
    'resnet50',
    bounding_box_format='xyxy',
    num_classes=10
)

Using TensorFlow backend


In [11]:
from tensorflow.keras.optimizers import Adam

# model compile
model.compile(optimizer=Adam(learning_rate=1e-4),
             classification_loss='focal',
             box_loss='smoothl1')

In [ ]:
import tensorflow as tf
# 모델 학습
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=400
)


Epoch 1/400
500/500 [==============================] - 32s 63ms/step - loss: 0.0000e+00 - box_loss: 0.0000e+00 - classification_loss: 0.0000e+00 - percent_boxes_matched_with_anchor: 0.4736 - val_loss: 0.0000e+00 - val_box_loss: 0.0000e+00 - val_classification_loss: 0.0000e+00 - val_percent_boxes_matched_with_anchor: 0.4790
Epoch 2/400
500/500 [==============================] - 30s 60ms/step - loss: 0.0000e+00 - box_loss: 0.0000e+00 - classification_loss: 0.0000e+00 - percent_boxes_matched_with_anchor: 0.4733 - val_loss: 0.0000e+00 - val_box_loss: 0.0000e+00 - val_classification_loss: 0.0000e+00 - val_percent_boxes_matched_with_anchor: 0.4790
Epoch 3/400
500/500 [==============================] - 30s 60ms/step - loss: 0.0000e+00 - box_loss: 0.0000e+00 - classification_loss: 0.0000e+00 - percent_boxes_matched_with_anchor: 0.4704 - val_loss: 0.0000e+00 - val_box_loss: 0.0000e+00 - val_classification_loss: 0.0000e+00 - val_percent_boxes_matched_with_anchor: 0.4852
Epoch 4/400
500/500 [====

In [ ]:
# 폴더 구조 만들기

In [ ]:
root_folder = "/home/govlept1004/jupyter_home/NUMBERdevkit"

In [ ]:
!mkdir $root_folder

In [ ]:
!mkdir $root_folder/Annotations
!mkdir $root_folder/JPEGImages
!mkdir $root_folder/ImageSets

In [ ]:
!mkdir $root_folder/ImageSets/Main

# CSV파일 PASCAL VOC로 변환하는 방법

In [ ]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
from PIL import Image

# 경로 설정
csv_path = "/home/govlept1004/jupyter_home/number_data.csv"
image_dir = "/home/govlept1004/jupyter_home/NUMBERdevkit/JPEGImages"
output_dir = "/home/govlept1004/jupyter_home/NUMBERdevkit/Annotations"
os.makedirs(output_dir, exist_ok=True)

# CSV 불러오기
df = pd.read_csv(csv_path)

# 파일 단위로 그룹화
grouped = df.groupby('filename')

# VOC 어노테이션 생성 함수
def create_voc_xml(filename, image_size, objects):
    annotation = ET.Element("annotation")
    
    ET.SubElement(annotation, "folder").text = os.path.basename(image_dir)
    ET.SubElement(annotation, "filename").text = filename
    
    size = ET.SubElement(annotation, "size")
    ET.SubElement(size, "width").text = str(image_size[0])
    ET.SubElement(size, "height").text = str(image_size[1])
    ET.SubElement(size, "depth").text = "3"

    for obj in objects:
        obj_element = ET.SubElement(annotation, "object")
        ET.SubElement(obj_element, "name").text = str(obj["label"])
        bndbox = ET.SubElement(obj_element, "bndbox")
        ET.SubElement(bndbox, "xmin").text = str(obj["left"])
        ET.SubElement(bndbox, "ymin").text = str(obj["top"])
        ET.SubElement(bndbox, "xmax").text = str(obj["left"] + obj["width"])
        ET.SubElement(bndbox, "ymax").text = str(obj["top"] + obj["height"])
    
    return ET.ElementTree(annotation)

# 각 이미지 파일에 대해 VOC XML 생성
for filename, group in grouped:
    image_path = os.path.join(image_dir, filename)
    image = Image.open(image_path)
    image_size = image.size  # (width, height)

    objects = group.to_dict("records")
    voc_xml = create_voc_xml(filename, image_size, objects)
    
    xml_filename = os.path.splitext(filename)[0] + ".xml"
    voc_xml.write(os.path.join(output_dir, xml_filename), encoding="utf-8", xml_declaration=True)


# train.txt / val.txt 20%씩 나눠서 자동 생성 코드

In [ ]:
import os
import random

# 경로 설정
annotations_dir = "/home/govlept1004/jupyter_home/NUMBERdevkit/Annotations"
imagesets_main_dir = "/home/govlept1004/jupyter_home/NUMBERdevkit/ImageSets/Main"
os.makedirs(imagesets_main_dir, exist_ok=True)

# 전체 파일 목록 로드
xml_files = [f for f in os.listdir(annotations_dir) if f.endswith(".xml")]

# 확장자 제거하고 숫자 기준 정렬
def get_numeric_id(filename):
    return int(os.path.splitext(filename)[0])

image_ids = sorted([os.path.splitext(f)[0] for f in xml_files], key=get_numeric_id)

# 셔플 후 분할
random.seed(42)  # 재현 가능성을 위해 고정 시드
random.shuffle(image_ids)

split_ratio = 0.8
split_index = int(len(image_ids) * split_ratio)
train_ids = image_ids[:split_index]
val_ids = image_ids[split_index:]

# 저장
with open(os.path.join(imagesets_main_dir, "train.txt"), "w") as f:
    f.writelines(f"{img_id}\n" for img_id in train_ids)

with open(os.path.join(imagesets_main_dir, "val.txt"), "w") as f:
    f.writelines(f"{img_id}\n" for img_id in val_ids)

print(f"✅ train/val.txt 생성 완료")
print(f"총 이미지 수: {len(image_ids)} → train: {len(train_ids)}개, val: {len(val_ids)}개")


# 폴더 구조가 만들어졌으니 이제 TFRecord로 변환시켜보아요!
제공된 코드를 사용하면되요!

In [ ]:
import os
import tensorflow as tf
import xml.etree.ElementTree as ET

# 클래스 이름을 문자열로 1~10까지 정확히 지정
VOC_CLASSES = [
    "1","2","3","4","5","6","7","8","9","0"
]
class_name_to_id = {name: i for i, name in enumerate(VOC_CLASSES)}

# XML 파싱 함수
def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.find("filename").text
    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    bboxes = []
    labels = []

    for obj in root.findall("object"):
        name = obj.find("name").text
        label = class_name_to_id[name]  # 문자열 그대로 매핑

        bbox = obj.find("bndbox")
        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))

        bboxes.append([xmin, ymin, xmax, ymax])
        labels.append(label)

    return filename, width, height, bboxes, labels

# TFRecord 예제 생성
def create_tf_example(xml_path, image_dir):
    filename, width, height, bboxes, labels = parse_voc_xml(xml_path)
    image_path = os.path.join(image_dir, filename)  # XML에 명시된 파일명을 그대로 사용

    with tf.io.gfile.GFile(image_path, 'rb') as fid:
        encoded_image = fid.read()

    xmins = [box[0] / width for box in bboxes]
    ymins = [box[1] / height for box in bboxes]
    xmaxs = [box[2] / width for box in bboxes]
    ymaxs = [box[3] / height for box in bboxes]

    classes_text = [VOC_CLASSES[label].encode('utf8') for label in labels]
    classes = labels

    feature = {
        'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[encoded_image])),
        'image/filename': tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename.encode('utf8')])),
        'image/height': tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
        'image/width': tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
        'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=xmins)),
        'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=ymins)),
        'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=xmaxs)),
        'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=ymaxs)),
        'image/object/class/text': tf.train.Feature(bytes_list=tf.train.BytesList(value=classes_text)),
        'image/object/class/label': tf.train.Feature(int64_list=tf.train.Int64List(value=classes)),
    }

    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example

# TFRecord 생성 함수
def generate_tfrecord(voc_dir, split_txt, output_path):
    annotation_dir = os.path.join(voc_dir, 'Annotations')
    image_dir = os.path.join(voc_dir, 'JPEGImages')

    with tf.io.TFRecordWriter(output_path) as writer:
        with open(split_txt, 'r') as f:
            lines = f.read().strip().splitlines()

        for img_id in lines:
            xml_path = os.path.join(annotation_dir, f"{img_id}.xml")

            if os.path.exists(xml_path):
                try:
                    example = create_tf_example(xml_path, image_dir)
                    writer.write(example.SerializeToString())
                except Exception as e:
                    print(f"[Error] {img_id} 처리 중 오류 발생: {e}")

VOC_ROOT_DIR = '/home/govlept1004/jupyter_home/NUMBERdevkit'

generate_tfrecord(
    voc_dir=VOC_ROOT_DIR,
    split_txt=os.path.join(VOC_ROOT_DIR, "ImageSets/Main/train.txt"),
    output_path=os.path.join(VOC_ROOT_DIR, "number_train.tfrecord")
)

generate_tfrecord(
    voc_dir=VOC_ROOT_DIR,
    split_txt=os.path.join(VOC_ROOT_DIR, "ImageSets/Main/val.txt"),
    output_path=os.path.join(VOC_ROOT_DIR, "number_val.tfrecord")
)


In [ ]:
# TFRecord가 생성 후 tf.data.Dataset 생성

IMAGE_SIZE = 128

def parse_tfrecord(example):
    features = {
        'image/encoded': tf.io.FixedLenFeature([], tf.string),
        'image/filename': tf.io.FixedLenFeature([], tf.string),
        'image/height': tf.io.FixedLenFeature([], tf.int64),
        'image/width': tf.io.FixedLenFeature([], tf.int64),
        'image/object/bbox/xmin': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/ymin': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/xmax': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/ymax': tf.io.VarLenFeature(tf.float32),
        'image/object/class/label': tf.io.VarLenFeature(tf.int64),
    }

    parsed = tf.io.parse_single_example(example, features)
    image = tf.image.decode_jpeg(parsed['image/encoded'], channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)

    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    xmins = tf.sparse.to_dense(parsed['image/object/bbox/xmin'])
    ymins = tf.sparse.to_dense(parsed['image/object/bbox/ymin'])
    xmaxs = tf.sparse.to_dense(parsed['image/object/bbox/xmax'])
    ymaxs = tf.sparse.to_dense(parsed['image/object/bbox/ymax'])
    labels = tf.sparse.to_dense(parsed['image/object/class/label'])

    boxes = tf.stack([xmins, ymins, xmaxs, ymaxs], axis=-1)

    return {
        "images": image,
        "bounding_boxes": {
            "boxes": boxes,
            "classes": tf.cast(labels, tf.int32),
        }
    }

def load_dataset(tfrecord_path, batch_size=8):
    dataset = tf.data.TFRecordDataset(tfrecord_path)
    dataset = dataset.map(parse_tfrecord, 
                          num_parallel_calls=tf.data.AUTOTUNE)
    
    # padded_batch 필수
    dataset = dataset.shuffle(1000).padded_batch(
        batch_size,
        padded_shapes={
            "images": [IMAGE_SIZE, IMAGE_SIZE, 3],
            "bounding_boxes": {
                "boxes": [None, 4],
                "classes": [None]
            }
        },
        drop_remainder=True
    ).prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = load_dataset("./NUMBERdevkit/number_train.tfrecord", 
                        batch_size=8)

val_ds = load_dataset("./NUMBERdevkit/number_val.tfrecord", 
                        batch_size=8)     

In [ ]:
# 재학습에 필요한 Dataset 준비 완료!!
# Model 생성
from keras_cv.models import RetinaNet

In [ ]:
model = RetinaNet.from_preset(
    'resnet50',
    bounding_box_format='xyxy',
    num_classes=10
)

In [ ]:
from tensorflow.keras.optimizers import Adam

# model compile
model.compile(optimizer=Adam(learning_rate=1e-4),
             classification_loss='focal',
             box_loss='smoothl1')

In [ ]:
import tensorflow as tf

# EarlyStopping 콜백 정의
earlystop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_classification_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1,
    mode='min'
)

# ModelCheckpoint
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath='/home/govlept1004/jupyter_home/NUMBERdevkit/best_retinanet_model_classification_loss.h5',
    monitor='val_classification_loss',
    save_best_only=True,
    save_weights_only=False,
    mode='min',
    verbose=1
)

# 모델 학습
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[earlystop_cb, checkpoint_cb]
)


In [ ]:
# 시각화 테스트

import os
import tensorflow as tf
import matplotlib.pyplot as plt

# 클래스 이름 (0~9 숫자)
VOC_CLASSES = [str(i) for i in range(10)]

# 이미지 로드 및 전처리 함수
def load_and_preprocess_image(image_path, image_size=128):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, [image_size, image_size])
    return image

# 시각화 함수
def draw_detections(image, boxes, scores, classes, class_names, threshold=0.5, top_k=5):
    plt.figure(figsize=(8,8))
    plt.imshow(image)
    ax = plt.gca()

    indices = tf.image.non_max_suppression(
        boxes, scores, max_output_size=top_k, iou_threshold=0.5, score_threshold=threshold
    )
    selected_boxes = tf.gather(boxes, indices).numpy()
    selected_scores = tf.gather(scores, indices).numpy()
    selected_classes = tf.gather(classes, indices).numpy()

    for i in range(selected_boxes.shape[0]):
        box = selected_boxes[i]
        class_id = int(selected_classes[i])
        score = selected_scores[i]

        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             fill=False, color="red", linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"{class_names[class_id]}: {score:.2f}", 
                color='red', fontsize=12, backgroundcolor='white')
    plt.axis("off")
    plt.show()

# 예측 대상 이미지 폴더
image_folder = "/home/govlept1004/jupyter_home/NUMBERdevkit/number_detect_image"
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for filename in image_files:
    image_path = os.path.join(image_folder, filename)
    image = load_and_preprocess_image(image_path)
    image_batch = tf.expand_dims(image, axis=0)  # (1, H, W, 3)

    predictions = model.predict(image_batch)

    boxes = predictions['boxes'][0]
    scores = predictions['confidence'][0]
    classes = predictions['classes'][0]

    print(f"🔍 Predicting {filename}...")
    draw_detections(image.numpy(), boxes, scores, classes, 
                    class_names=VOC_CLASSES, 
                    threshold=0.5, top_k=2)


In [ ]:
# RetinaNet 모델을 저장한 후, 중단된 위치에서 정확히 이어서 재학습하는 코드 예시

# 1단계: 최초 학습 시 저장
from keras.callbacks import ModelCheckpoint, EarlyStopping

# 전체 모델 저장
checkpoint = ModelCheckpoint(
    "retinanet_full_model.keras",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# 학습 시작
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
# 2단계: 학습 재개할 때 (중단된 후)
from keras_cv.models import RetinaNet

# 커스텀 객체 등록
custom_objects = {"RetinaNet": RetinaNet}

# 저장된 전체 모델 로드
model = tf.keras.models.load_model(
    "retinanet_full_model.keras",
    custom_objects=custom_objects
)

# 이어서 추가 학습
model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=model.optimizer.iterations.numpy() // len(train_ds), # initial_epoch 없이 바로 .fit() 하면 0부터 다시 카운팅됨
    epochs=150,  # 총 학습하고 싶은 epoch
    callbacks=[checkpoint, early_stopping]
)
